# 2. EDA


In [ ]:
from matplotlib import pyplot as plt 
import seaborn as sns 


In [ ]:
df_train = spark.read.format("delta").load("/Volumes/workspace/default/data_customers/bronze/train")
df_test = spark.read.format("delta").load("/Volumes/workspace/default/data_customers/bronze/test")


In [ ]:
df_train = df_train.drop("CustomerID")
df_test = df_test.drop("CustomerID")


In [ ]:
df_train.printSchema()


In [ ]:
df_test.printSchema()


In [ ]:
from pyspark.sql.functions import col, sum, when

missing_df_train= df_train.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df_train.columns
])

display(missing_df_train)


In [ ]:
df_train = df_train.dropna()
df_test = df_test.dropna()


df_train.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df_train.columns
]).show()


In [ ]:
df_train = df_train.dropDuplicates()
df_test = df_test.dropDuplicates()


In [ ]:
for c in df_train.columns:
    print(c, df_train.select(c).distinct().count())


In [ ]:
display(df_train.describe())


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

churn_pd = df_train.select("Churn").toPandas()

plt.figure(figsize=(6,4))
ax = sns.countplot(x='Churn', data=churn_pd)

for p in ax.patches:
    ax.annotate(
        f'{int(p.get_height())}',
        (p.get_x() + p.get_width()/2., p.get_height()),
        ha='center',
        va='bottom'
    )

plt.show()


In [ ]:
num_cols = [
    f.name for f in df_train.schema.fields
    if f.dataType.simpleString() in ["int", "double"]
    and f.name != "Churn"
]

print("Numeric columns:", num_cols)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

for col in num_cols:
    
    data = df_train.select(col).toPandas()[col]
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].hist(data, bins=30, density=True, alpha=0.7)
    axes[0].set_title(f"Histogram of {col}")
    axes[0].set_xlabel(col)
    axes[0].set_ylabel("Density")
    axes[0].grid(alpha=0.3)

    sns.kdeplot(data, fill=True, ax=axes[1], color='orange')
    axes[1].set_title(f"KDE of {col}")
    axes[1].set_xlabel(col)
    axes[1].set_ylabel("Density")
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sample_df = df_train.sample(fraction=0.2, seed=42)

for col in num_cols:
    
    data = sample_df.select(col).toPandas()[col]

    fig, axes = plt.subplots(1, 2, figsize=(12,4))

    axes[0].boxplot(
        data,
        vert=False,
        patch_artist=True,
        boxprops=dict(facecolor='skyblue', alpha=0.7),
        medianprops=dict(color='red')
    )
    axes[0].set_title(f"Boxplot of {col}")
    axes[0].set_xlabel(col)
    axes[0].grid(axis='x', alpha=0.3)

    sns.violinplot(
        x=data,
        ax=axes[1],
        inner='quartile',
        color='orange'
    )
    axes[1].set_title(f"Violin Plot of {col}")
    axes[1].set_xlabel(col)
    axes[1].grid(axis='x', alpha=0.3)

    plt.tight_layout()
    plt.show()


In [ ]:
from pyspark.sql.functions import avg

churn_rate = (
    df_train
    .groupBy("Support Calls")
    .agg(avg("Churn").alias("churn_rate"))
    .orderBy("Support Calls")
)

churn_pd = churn_rate.toPandas()

plt.figure(figsize=(8,5))
plt.bar(churn_pd['Support Calls'], churn_pd['churn_rate'])

plt.xlabel("Number of Support Calls")
plt.ylabel("Churn Rate")
plt.title("Churn Rate by Support Calls")
plt.ylim(0,1)

plt.show()


In [ ]:
cols = num_cols + ["Churn"]
pdf = df_train.select(cols).toPandas()

corr = pdf.corr()

plt.figure(figsize=(10,8))
sns.heatmap(corr, annot=True, cmap='coolwarm')
plt.title("Correlation Matrix including Churn")
plt.show()


### Check Categorical Columns


In [ ]:
categorical_cols = [f.name for f in df_train.schema.fields 
                    if f.dataType.simpleString() == 'string']
categorical_cols


In [ ]:
for col in categorical_cols:
    print(f"\n===== {col} =====")
    df_train.groupBy(col).count().orderBy("count", ascending=False).show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

pdf = df_train.select(categorical_cols).toPandas()

for col in categorical_cols:
    plt.figure(figsize=(6,4))
    
    sns.countplot(
        x=pdf[col],
        order=pdf[col].value_counts().index
    )
    
    plt.title(f"{col} Distribution")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


### Data Engineering : New Column


In [ ]:
from pyspark.sql.functions import when, col

df_train = df_train.withColumn(
    "Age_group",
    when(col("Age") < 30, "Young (<30)")
    .when((col("Age") >= 30) & (col("Age") < 50), "Adult (30-50)")
    .otherwise("Senior (50+)")
)

df_test = df_test.withColumn(
    "Age_group",
    when(col("Age") < 30, "Young (<30)")
    .when((col("Age") >= 30) & (col("Age") < 50), "Adult (30-50)")
    .otherwise("Senior (50+)")
)


In [ ]:
import matplotlib.pyplot as plt

pdf = df_train.select("Age_group", "Churn").toPandas()

pivot = (
    pdf.groupby(["Age_group", "Churn"])
       .size()
       .unstack(fill_value=0)
)

plt.figure(figsize=(12,6))
pivot.plot(kind='bar')
plt.title('Age Group vs Churn')
plt.xlabel('Age Group')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.legend(title='Churn')
plt.tight_layout()
plt.show()


### Check Last Interaction with Churn


In [ ]:
import matplotlib.pyplot as plt

pdf = df_train.select("Last Interaction", "Churn").toPandas()

pivot = (
    pdf.groupby(["Last Interaction", "Churn"])
       .size()
       .unstack(fill_value=0)
)

pivot_norm = pivot.div(pivot.sum(axis=1), axis=0)

plt.figure(figsize=(12,6))
pivot_norm.plot(kind='bar', stacked=True)

plt.title("Percentage Distribution of Churn by Last Interaction")
plt.ylabel("Percentage")
plt.xlabel("Last Interaction")
plt.legend(title="Churn")
plt.tight_layout()
plt.show()
